# 08. Sliding Window Merge (슬라이딩 윈도우 + 통합)

## 목적
1. 코호트 기준 슬라이딩 윈도우 생성
2. 모든 raw 테이블을 윈도우 기준으로 집계 & 병합
3. 레이블 생성 (death, vent, pressor)

## 입력 (Raw 테이블들)
- `cohort_base.csv`: 기본 코호트
- `vital_raw.csv`: 활력징후
- `lab_raw.csv`: 검사 결과
- `ventilation_raw.csv`: 인공호흡기
- `pressor_raw.csv`: 승압제
- `urine_raw.csv`: 소변량
- `gcs_raw.csv`: GCS 점수

## 출력
- `sliding_window_merged.csv`: 슬라이딩 윈도우 통합 데이터

## 윈도우 설정
- **Window Size**: 6시간
- **Stride**: 1시간
- **Range**: ICU 입실 후 6h ~ 72h
- **예상 결과**: ~93만 rows (약 23K 환자 × 약 40 시점)

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import timedelta

# 설정
INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

# 윈도우 파라미터
WINDOW_SIZE_H = 6   # 윈도우 크기 (시간)
STRIDE_H = 1        # 이동 간격 (시간)
MIN_HOUR = 6        # 시작 시점 (ICU 입실 후)
MAX_HOUR = 72       # 종료 시점

print("=== 08. Sliding Window Merge 시작 ===")
print(f"\n윈도우 설정:")
print(f"  - Window Size: {WINDOW_SIZE_H}h")
print(f"  - Stride: {STRIDE_H}h")
print(f"  - Range: {MIN_HOUR}h ~ {MAX_HOUR}h")

## Step 1: 코호트 및 Raw 테이블 로드

In [ ]:
print("\nStep 1: 데이터 로드")

# 코호트
df_cohort = pd.read_csv(
    os.path.join(INPUT_DIR, 'cohort_base.csv'),
    parse_dates=['intime', 'outtime', 'deathtime', 'dnr_time', 'vent_start_time', 'pressor_start_time']
)
print(f"✓ cohort_base: {len(df_cohort):,}명")

# Raw 테이블들
df_vital = pd.read_csv(os.path.join(INPUT_DIR, 'vital_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ vital_raw: {len(df_vital):,} rows")

df_lab = pd.read_csv(os.path.join(INPUT_DIR, 'lab_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ lab_raw: {len(df_lab):,} rows")

df_vent = pd.read_csv(os.path.join(INPUT_DIR, 'ventilation_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ ventilation_raw: {len(df_vent):,} rows")

df_pressor = pd.read_csv(os.path.join(INPUT_DIR, 'pressor_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ pressor_raw: {len(df_pressor):,} rows")

df_urine = pd.read_csv(os.path.join(INPUT_DIR, 'urine_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ urine_raw: {len(df_urine):,} rows")

df_gcs = pd.read_csv(os.path.join(INPUT_DIR, 'gcs_raw.csv'), parse_dates=['charttime_h'])
print(f"✓ gcs_raw: {len(df_gcs):,} rows")

## Step 2: 슬라이딩 윈도우 생성

In [ ]:
print("\nStep 2: 슬라이딩 윈도우 생성")

# 각 환자별 윈도우 생성
windows = []

for _, row in df_cohort.iterrows():
    stay_id = row['stay_id']
    intime = row['intime']
    outtime = row['outtime']
    
    # 6h부터 72h까지 1시간 간격
    for obs_hour in range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H):
        obs_end = intime + timedelta(hours=obs_hour)
        obs_start = intime + timedelta(hours=obs_hour - WINDOW_SIZE_H)
        
        # ICU 체류 중인 경우만
        if obs_end <= outtime:
            windows.append({
                'stay_id': stay_id,
                'observation_hour': obs_hour,
                'observation_start': obs_start,
                'observation_end': obs_end
            })

df_windows = pd.DataFrame(windows)
print(f"✓ 초기 윈도우 생성: {len(df_windows):,}개")

# 코호트 정보 병합
df_windows = df_windows.merge(
    df_cohort[['stay_id', 'subject_id', 'hadm_id', 'anchor_age', 'gender', 
               'first_careunit', 'deathtime', 'dnr_time', 'vent_start_time', 
               'pressor_start_time', 'icu_mortality', 'hospital_mortality', 'is_readmission']],
    on='stay_id',
    how='left'
)
print(f"✓ 코호트 정보 병합 완료")

## Step 3: Event Censoring(Death) 적용

In [ ]:
# print("\nStep 3: Event Censoring(Death) 적용")

# before_filter = len(df_windows)

# # 각 필터별 감소량 확인
# print(f"전체: {before_filter:,}")

# # DNR만
# temp = df_windows[(df_windows['dnr_time'].isna()) | 
#                   (df_windows['observation_end'] < df_windows['dnr_time'])]
# print(f"DNR 후: {len(temp):,}")

# # Death만  
# temp = df_windows[(df_windows['deathtime'].isna()) | 
#                   (df_windows['observation_end'] < df_windows['deathtime'])]
# print(f"Death 후: {len(temp):,}")

# # Vent만
# temp = df_windows[(df_windows['vent_start_time'].isna()) | 
#                   (df_windows['observation_end'] < df_windows['vent_start_time'])]
# print(f"Vent 후: {len(temp):,}")

# # Pressor만
# temp = df_windows[(df_windows['pressor_start_time'].isna()) | 
#                   (df_windows['observation_end'] < df_windows['pressor_start_time'])]
# print(f"Pressor 후: {len(temp):,}")

In [ ]:
print("\nStep 3: Death Censoring 적용")

before_filter = len(df_windows)

# 사망 이전 윈도우만 유지 (observation_end 기준)
df_windows = df_windows[
    (df_windows['deathtime'].isna()) |
    (df_windows['observation_end'] < df_windows['deathtime'])
]

after_filter = len(df_windows)
print(f"✓ Censoring 적용: {before_filter:,} → {after_filter:,} ({after_filter/before_filter*100:.1f}%)")

## Step 4: 레이블 생성

In [ ]:
print("\nStep 4: 레이블 생성")

# 6h, 12h, 24h 예측 레이블
for horizon in [6, 12, 24]:
    horizon_delta = timedelta(hours=horizon)
    
    # Death
    df_windows[f'death_next_{horizon}h'] = (
        (df_windows['deathtime'].notna()) &
        (df_windows['deathtime'] > df_windows['observation_end']) &
        (df_windows['deathtime'] <= df_windows['observation_end'] + horizon_delta)
    ).astype(int)
    
    # Ventilation
    df_windows[f'vent_next_{horizon}h'] = (
        (df_windows['vent_start_time'].notna()) &
        (df_windows['vent_start_time'] > df_windows['observation_end']) &
        (df_windows['vent_start_time'] <= df_windows['observation_end'] + horizon_delta)
    ).astype(int)
    
    # Pressor
    df_windows[f'pressor_next_{horizon}h'] = (
        (df_windows['pressor_start_time'].notna()) &
        (df_windows['pressor_start_time'] > df_windows['observation_end']) &
        (df_windows['pressor_start_time'] <= df_windows['observation_end'] + horizon_delta)
    ).astype(int)
    
    # Composite (any event)
    df_windows[f'composite_next_{horizon}h'] = (
        (df_windows[f'death_next_{horizon}h'] == 1) |
        (df_windows[f'vent_next_{horizon}h'] == 1) |
        (df_windows[f'pressor_next_{horizon}h'] == 1)
    ).astype(int)

print("\n=== 레이블 분포 ===")
for horizon in [6, 12, 24]:
    print(f"\n{horizon}h 예측:")
    print(f"  - Death: {df_windows[f'death_next_{horizon}h'].sum():,} ({df_windows[f'death_next_{horizon}h'].mean()*100:.2f}%)")
    print(f"  - Vent: {df_windows[f'vent_next_{horizon}h'].sum():,} ({df_windows[f'vent_next_{horizon}h'].mean()*100:.2f}%)")
    print(f"  - Pressor: {df_windows[f'pressor_next_{horizon}h'].sum():,} ({df_windows[f'pressor_next_{horizon}h'].mean()*100:.2f}%)")
    print(f"  - Composite: {df_windows[f'composite_next_{horizon}h'].sum():,} ({df_windows[f'composite_next_{horizon}h'].mean()*100:.2f}%)")

## Step 5: 피처 테이블 병합 (윈도우 기준 집계)

In [ ]:
print("\nStep 5: 피처 테이블 병합")

def aggregate_to_window(df_raw, df_windows, feature_cols, agg_func='mean'):
    """
    Raw 데이터를 윈도우 기준으로 집계
    """
    results = []
    
    for _, window in df_windows.iterrows():
        stay_id = window['stay_id']
        obs_start = window['observation_start']
        obs_end = window['observation_end']
        
        # 해당 윈도우 내 데이터 필터링
        mask = (
            (df_raw['stay_id'] == stay_id) &
            (df_raw['charttime_h'] >= obs_start) &
            (df_raw['charttime_h'] <= obs_end)
        )
        window_data = df_raw.loc[mask, feature_cols]
        
        if len(window_data) > 0:
            if agg_func == 'mean':
                agg_values = window_data.mean()
            elif agg_func == 'max':
                agg_values = window_data.max()
            elif agg_func == 'min':
                agg_values = window_data.min()
            elif agg_func == 'last':
                agg_values = window_data.iloc[-1]
            results.append(agg_values.to_dict())
        else:
            results.append({col: np.nan for col in feature_cols})
    
    return pd.DataFrame(results)

# 더 효율적인 방식: merge_asof 또는 groupby 활용
# 여기서는 간단한 버전으로 구현

print("  피처 집계 중... (시간이 걸릴 수 있음)")
print("  → 실제 운영 시에는 SQL 또는 벡터화된 방식 권장")

In [ ]:
# Vital 항목별 집계 함수 적용 + min/max 통계 추가
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

df_vital_merged = df_vital.merge(
    df_cohort[['stay_id', 'intime']], 
    on='stay_id', 
    how='left'
)
df_vital_merged['hours_since_admit'] = (
    (df_vital_merged['charttime_h'] - df_vital_merged['intime']).dt.total_seconds() / 3600
).round().astype(int)

vital_agg = []
obs_hours = list(range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H))

for obs_hour in tqdm(obs_hours, desc="Vital 집계"):
    window_start = obs_hour - WINDOW_SIZE_H
    window_end = obs_hour
    
    mask = (
        (df_vital_merged['hours_since_admit'] > window_start) &
        (df_vital_merged['hours_since_admit'] <= window_end)
    )
    
    window_data = df_vital_merged[mask]
    
    agg_dict = {}
    for stay_id, grp in window_data.groupby('stay_id'):
        row = {
            'hr': grp['hr'].median() if grp['hr'].notna().any() else np.nan,
            'rr': grp['rr'].median() if grp['rr'].notna().any() else np.nan,
            'spo2': grp['spo2'].min() if grp['spo2'].notna().any() else np.nan,
            'sbp': grp['sbp'].min() if grp['sbp'].notna().any() else np.nan,
            'dbp': grp['dbp'].min() if grp['dbp'].notna().any() else np.nan,
            'mbp': grp['mbp'].min() if grp['mbp'].notna().any() else np.nan,
            'temp': grp.loc[grp['temp'].notna(), 'temp'].iloc[-1] if grp['temp'].notna().any() else np.nan,
            'hr_max': grp['hr'].max() if grp['hr'].notna().any() else np.nan,
            'rr_max': grp['rr'].max() if grp['rr'].notna().any() else np.nan,
            'spo2_min': grp['spo2'].min() if grp['spo2'].notna().any() else np.nan,
            'sbp_min': grp['sbp'].min() if grp['sbp'].notna().any() else np.nan,
        }
        agg_dict[stay_id] = row
    
    agg = pd.DataFrame.from_dict(agg_dict, orient='index').reset_index()
    agg.columns = ['stay_id', 'hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp', 'temp', 
                   'hr_max', 'rr_max', 'spo2_min', 'sbp_min']
    agg['observation_hour'] = obs_hour
    vital_agg.append(agg)

df_vital_agg = pd.concat(vital_agg, ignore_index=True)
print(f"\n✓ Vital 집계 완료: {len(df_vital_agg):,} rows")

df_windows = df_windows.merge(
    df_vital_agg,
    on=['stay_id', 'observation_hour'],
    how='left'
)
print(f"✓ Vital 병합 완료")

In [ ]:
# Lab 항목별 집계 함수 적용
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

df_lab_merged = df_lab.merge(
    df_cohort[['stay_id', 'intime']], 
    on='stay_id', 
    how='left'
)
df_lab_merged['hours_since_admit'] = (
    (df_lab_merged['charttime_h'] - df_lab_merged['intime']).dt.total_seconds() / 3600
).round().astype(int)

# 항목별 집계 함수 정의
lab_cols = ['sao2', 'ph', 'lactate', 'creatinine', 'bilirubin', 'wbc', 'platelets', 'potassium', 'sodium']

# Lactate 이상치 클리핑 (상한 20 mmol/L)
df_lab_merged['lactate'] = df_lab_merged['lactate'].clip(upper=20)

lab_agg = []
obs_hours = list(range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H))

for obs_hour in tqdm(obs_hours, desc="Lab 집계"):
    window_start = obs_hour - 24  # 24시간 이내 검사 사용
    window_end = obs_hour
    
    mask = (
        (df_lab_merged['hours_since_admit'] > window_start) &
        (df_lab_merged['hours_since_admit'] <= window_end)
    )
    
    window_data = df_lab_merged[mask]
    
    # 항목별 집계
    agg_dict = {}
    for stay_id, grp in window_data.groupby('stay_id'):
        agg_dict[stay_id] = {
            'sao2': grp.loc[grp['sao2'] > 0, 'sao2'].min() if (grp['sao2'] > 0).any() else np.nan,
            'ph': grp.loc[grp['ph'] > 0, 'ph'].min() if (grp['ph'] > 0).any() else np.nan,
            'lactate': grp['lactate'].max(),
            'creatinine': grp['creatinine'].max(),
            'bilirubin': grp['bilirubin'].max(),
            'wbc': grp['wbc'].median(),
            'platelets': grp['platelets'].median(),
            'potassium': grp['potassium'].median(),
            'sodium': grp['sodium'].median(),
        }
    
    agg = pd.DataFrame.from_dict(agg_dict, orient='index').reset_index()
    agg.columns = ['stay_id'] + lab_cols
    agg['observation_hour'] = obs_hour
    lab_agg.append(agg)

df_lab_agg = pd.concat(lab_agg, ignore_index=True)
print(f"\n✓ Lab 집계 완료: {len(df_lab_agg):,} rows")

df_windows = df_windows.merge(
    df_lab_agg,
    on=['stay_id', 'observation_hour'],
    how='left'
)
print(f"✓ Lab 병합 완료")

In [ ]:
# GCS 병합 - MIN (최저값 = 최악의 의식 상태)
df_gcs_merged = df_gcs.merge(
    df_cohort[['stay_id', 'intime']], 
    on='stay_id', 
    how='left'
)
df_gcs_merged['hours_since_admit'] = (
    (df_gcs_merged['charttime_h'] - df_gcs_merged['intime']).dt.total_seconds() / 3600
).round().astype(int)

gcs_cols = ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']

gcs_agg = []
for obs_hour in range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H):
    window_start = obs_hour - WINDOW_SIZE_H
    window_end = obs_hour
    
    mask = (
        (df_gcs_merged['hours_since_admit'] > window_start) &
        (df_gcs_merged['hours_since_admit'] <= window_end)
    )
    
    # MIN 사용 (낮을수록 나쁨)
    agg = df_gcs_merged[mask].groupby('stay_id')[gcs_cols].min().reset_index()
    agg['observation_hour'] = obs_hour
    gcs_agg.append(agg)

df_gcs_agg = pd.concat(gcs_agg, ignore_index=True)
print(f"✓ GCS 집계 완료: {len(df_gcs_agg):,} rows")

df_windows = df_windows.merge(
    df_gcs_agg,
    on=['stay_id', 'observation_hour'],
    how='left'
)
print(f"✓ GCS 병합 완료")

In [ ]:
# Urine 병합
df_urine_merged = df_urine.merge(
    df_cohort[['stay_id', 'intime']], 
    on='stay_id', 
    how='left'
)
df_urine_merged['hours_since_admit'] = (
    (df_urine_merged['charttime_h'] - df_urine_merged['intime']).dt.total_seconds() / 3600
).round().astype(int)

urine_cols = ['urine_ml', 'urine_ml_kg_hr', 'oliguria_flag']

urine_agg = []
for obs_hour in range(MIN_HOUR, MAX_HOUR + 1, STRIDE_H):
    window_start = obs_hour - WINDOW_SIZE_H
    window_end = obs_hour
    
    mask = (
        (df_urine_merged['hours_since_admit'] > window_start) &
        (df_urine_merged['hours_since_admit'] <= window_end)
    )
    
    # 합계와 평균
    agg = df_urine_merged[mask].groupby('stay_id').agg({
        'urine_ml': 'sum',
        'urine_ml_kg_hr': 'mean',
        'oliguria_flag': 'max'
    }).reset_index()
    agg.columns = ['stay_id', 'urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag']
    agg['observation_hour'] = obs_hour
    urine_agg.append(agg)

df_urine_agg = pd.concat(urine_agg, ignore_index=True)
print(f"✓ Urine 집계 완료: {len(df_urine_agg):,} rows")

df_windows = df_windows.merge(
    df_urine_agg,
    on=['stay_id', 'observation_hour'],
    how='left'
)
print(f"✓ Urine 병합 완료")

## Step 6: 결과 확인 및 저장

In [ ]:
print("\n" + "="*60)
print("최종 결과 요약")
print("="*60)

print(f"\n총 행 수: {len(df_windows):,}개")
print(f"고유 stay: {df_windows['stay_id'].nunique():,}개")
print(f"고유 환자: {df_windows['subject_id'].nunique():,}명")
print(f"환자당 평균 윈도우: {len(df_windows) / df_windows['stay_id'].nunique():.1f}개")

print(f"\n=== 컬럼 목록 ({len(df_windows.columns)}개) ===")
for col in df_windows.columns:
    missing = df_windows[col].isna().mean() * 100
    print(f"  - {col}: {missing:.1f}% 결측")

In [ ]:
# 저장
output_path = os.path.join(OUTPUT_DIR, 'sliding_window_merged.csv')
df_windows.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: sliding_window_merged.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 경로: {output_path}")

In [ ]:
print("\n=== 샘플 데이터 ===")
df_windows.head()

In [ ]:
print("\n=== 08. Sliding Window Merge 완료 ===")